In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ["OMP_NUM_THREADS"] = '1'
import torch
import pickle
from vmc_torch.experiment.vmap.vmap_torch_utils import RobustSVD
import autoray as ar
import symmray as sr
import quimb.tensor as qtn
from vmc_torch.GPU.GPU_vmap_utils import random_initial_config
from torch import nn
from vmc_torch.experiment.vmap.vmap_models import fPEPS_Model


# 设置默认精度
dtype = torch.float64
torch.set_default_dtype(dtype)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device_cpu = torch.device('cpu')

JITTER = 1e-10
# driver='gesvd' if device.type == 'cuda' else None
driver = None
ar.register_function('torch', 'linalg.svd', lambda x, **kwargs: RobustSVD.apply(x, JITTER, driver))
# ==========================================
# 2. 参数设置与模型加载
# ==========================================
Lx, Ly = 8, 8
nsites = Lx * Ly
N_f = nsites - 8
D = 8
chi = 8

# 路径配置 (保持你的原样)
pwd = '/home/sijingdu/TNVMC/VMC_code/vmc_torch/vmc_torch/experiment/vmap/data'
u1z2 = True
appendix = '_U1SU' if u1z2 else ''

# 加载骨架 (这部分很快，可以在 CPU 做完再转 GPU)
# 注意：pickle load 最好只在 Rank 0 做然后广播，或者大家各自读文件(如果文件系统支持并发)
# 这里假设大家各自读文件没问题
params_pkl = pickle.load(open(pwd+f'/{Lx}x{Ly}/t=1.0_U=8.0/N={N_f}/Z2/D={D}/peps_su_params{appendix}.pkl', 'rb'))
skeleton = pickle.load(open(pwd+f'/{Lx}x{Ly}/t=1.0_U=8.0/N={N_f}/Z2/D={D}/peps_skeleton{appendix}.pkl', 'rb'))
peps = qtn.unpack(params_pkl, skeleton)

# 预处理 (CPU)
for ts in peps.tensors:
    ts.modify(data=ts.data.to_flat() * 4)
for site in peps.sites:
    peps[site].data._label = site
    peps[site].data.indices[-1]._linearmap = ((0, 0), (1, 0), (1, 1), (0, 1))


# 初始化模型并移动到 GPU
fpeps_model = fPEPS_Model(
    tn=peps, max_bond=chi, dtype=dtype,
    contract_boundary_opts={
        'mode': 'mps',
        'equalize_norms': 1.0,
        'canonize': True,
    }
)
fpeps_model.to(device) # <--- 关键：模型全在 GPU


# ==========================================
# 3. 采样配置
# ==========================================

# 并行运行的 Chain 数量 (Batch Size)
# 如果显存够，可以直接设为 samples_per_rank，这样一步到位
# 如果显存不够，可以设小一点，循环多次累积
batch_size_per_rank = 1024
# 确保初始化 walkers 在 GPU 上
fxs_list = [random_initial_config(N_f, nsites, seed=42+_) for _ in range(batch_size_per_rank)]
fxs = torch.stack(fxs_list).to(device)

In [26]:
%%time
with torch.no_grad():
    amps = fpeps_model(fxs)

CPU times: user 10 s, sys: 4.06 s, total: 14.1 s
Wall time: 15.7 s


In [27]:
fpeps_model_cpu = fPEPS_Model(
    tn=peps, max_bond=chi, dtype=dtype,
    contract_boundary_opts={
        'mode': 'mps',
        'equalize_norms': 1.0,
        'canonize': True,
    }
)
fpeps_model_cpu.to(device_cpu) # <--- 关键：模型全在 CPU
fxs_cpu = fxs.to(device_cpu)

In [28]:
%%time
with torch.no_grad():
    amps_cpu = fpeps_model_cpu(fxs_cpu)

CPU times: user 24.6 s, sys: 31.1 s, total: 55.8 s
Wall time: 55.7 s


In [4]:
# 尝试编译模型
c_fpeps_model = fPEPS_Model(
    tn=peps, max_bond=chi, dtype=dtype, compile=True
)

In [5]:
with torch.no_grad():
    c_fpeps_model(fxs)

In [6]:
%%time
with torch.no_grad():
    c_fpeps_model(fxs)

CPU times: user 10.6 s, sys: 4.07 s, total: 14.7 s
Wall time: 16.4 s
